# Data Cleaning & Preparation
**Mobile Money Transaction Analysis Project**

This notebook implements the complete data cleaning pipeline:
1. Data Extraction (both original and anonymized datasets)
2. Data Cleaning (applied separately to each dataset)
3. Data Source Labeling
4. Dataset Validation & Comparison
5. Merging
6. Feature Engineering
7. Final Output

## 0. Setup & Imports

In [1]:
import re
import csv
import hashlib
import os
import sys
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 30)

BASE_DIR = Path(os.getcwd()).resolve()
if BASE_DIR.name == '2_Data_Cleaning':
    BASE_DIR = BASE_DIR.parent

ORIGINAL_RAW_DIR = BASE_DIR / "Data" / "Original_Raw"
ANONYMIZED_RAW_DIR = BASE_DIR / "Data" / "Anonymized_raw"
EXTRACTED_ORIGINAL_DIR = BASE_DIR / "Data" / "extracted" / "original"
EXTRACTED_ANON_DIR = BASE_DIR / "Data" / "extracted" / "anonymized"
CLEANED_DIR = BASE_DIR / "Data" / "cleaned"
OUTPUT_DIR = BASE_DIR / "Data" / "output"

for d in [EXTRACTED_ORIGINAL_DIR, EXTRACTED_ANON_DIR, CLEANED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Original raw files: {len(list(ORIGINAL_RAW_DIR.iterdir()))}")
print(f"Anonymized raw files: {len(list(ANONYMIZED_RAW_DIR.iterdir()))}")

Base directory: C:\Users\ghisl\Documents\DataScience_Final_Project
Original raw files: 51
Anonymized raw files: 18


## 1. Extractor Functions (from Mobile_Money_Data_Extractor V2-1)

The following functions are copied directly from the lecturer-provided extractor notebook. They handle:
- File loading and column normalization
- Operator detection (OrangeMoney / MobileMoney)
- Balance-message filtering
- Transaction amount and balance extraction
- Transaction type classification
- Message anonymization

In [2]:
# ── Column name aliases (FR / EN) from extractor notebook ──
COL_ALIASES = {
    'date':      ['date'],
    'heure':     ['heure', 'time'],
    'direction': ['direction'],
    'contact':   ['contact'],
    'telephone': ['téléphone', 'telephone', 'phone'],
    'contenu':   ['contenu', 'content'],
    'type':      ['type'],
}

def normalize_columns(df):
    """Remap any FR or EN column names to canonical lowercase names."""
    rename = {}
    for canonical, aliases in COL_ALIASES.items():
        for col in df.columns:
            if str(col).strip().lower() in aliases:
                rename[col] = canonical
                break
    return df.rename(columns=rename)

def load_file(path):
    """Load an OrangeMoney or MobileMoney Excel/CSV export.
    Auto-detects the header row (some files have metadata rows at top)."""
    path = str(path)
    if path.endswith('.xlsx'):
        for skip in range(5):
            df = pd.read_excel(path, skiprows=skip)
            cols_lower = [str(c).strip().lower() for c in df.columns]
            if any(c in cols_lower for c in ['date', 'contenu', 'content']):
                df = normalize_columns(df)
                if 'contenu' in df.columns:
                    df = df.dropna(subset=['contenu'], how='all')
                    df['contenu'] = df['contenu'].astype(str).str.replace('_x000d_', ' ', regex=False).str.strip()
                return df
    elif path.endswith('.csv'):
        for skip in range(5):
            try:
                df = pd.read_csv(path, skiprows=skip)
                cols_lower = [str(c).strip().lower() for c in df.columns]
                if any(c in cols_lower for c in ['date', 'contenu', 'content']):
                    df = normalize_columns(df)
                    if 'contenu' in df.columns:
                        df = df.dropna(subset=['contenu'], how='all')
                        df['contenu'] = df['contenu'].astype(str).str.replace('_x000d_', ' ', regex=False).str.strip()
                    return df
            except Exception:
                continue
    raise ValueError(f'Could not detect column headers in: {path}')


def load_anonymized_csv(path):
    """Parse the pre-anonymized CSV files (user011-user018) with non-standard quoting."""
    rows = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader)
            for row in reader:
                if len(row) >= 4:
                    rows.append({
                        'date': row[0].strip(),
                        'heure': row[1].strip(),
                        'contact': row[2].strip(),
                        'contenu': row[3].strip().strip('"'),
                    })
                elif len(row) == 1 and ',' in row[0]:
                    inner = row[0].strip().strip('"').replace('""', '"')
                    parts = inner.split(',', 3)
                    if len(parts) >= 4:
                        rows.append({
                            'date': parts[0].strip(),
                            'heure': parts[1].strip(),
                            'contact': parts[2].strip(),
                            'contenu': parts[3].strip().strip('"'),
                        })
    except Exception:
        pass

    if not rows:
        with open(path, 'r', encoding='utf-8') as f:
            raw = f.read()
        lines = raw.strip().split('\n')
        current_row = None
        for line in lines[1:]:
            line_stripped = line.strip()
            if not line_stripped:
                continue
            if line_stripped.startswith('"') and current_row is None:
                current_row = line_stripped
            elif current_row is not None:
                current_row += ' ' + line_stripped
            if current_row and current_row.endswith('"'):
                inner = current_row[1:-1].replace('""', '"')
                parts = inner.split(',', 3)
                if len(parts) >= 4:
                    rows.append({
                        'date': parts[0].strip(),
                        'heure': parts[1].strip(),
                        'contact': parts[2].strip(),
                        'contenu': parts[3].strip().strip('"'),
                    })
                current_row = None

    if not rows:
        return pd.DataFrame(columns=['date', 'heure', 'contact', 'contenu'])
    return pd.DataFrame(rows)

print("File loaders ready.")

File loaders ready.


In [3]:
# ── Operator detection ──
def detect_operator(df):
    for col in ['contact', 'telephone']:
        if col in df.columns:
            sample = df[col].dropna().astype(str).str.lower()
            if sample.str.contains('orangemoney').any():
                return 'OrangeMoney'
            if sample.str.contains('mobilemoney').any():
                return 'MobileMoney'
    return 'Unknown'

def detect_operator_from_content(df):
    if 'contenu' in df.columns:
        sample = df['contenu'].dropna().astype(str).str.lower()
        if sample.str.contains('orange').any():
            return 'OrangeMoney'
        if sample.str.contains('momo|mtn|mobile money').any():
            return 'MobileMoney'
    return 'Unknown'

# ── Balance-message filter ──
BALANCE_KEYWORDS = [
    'nouveau solde', 'nouveau solde est',
    'new balance', 'your new balance',
]
BALANCE_RE_FILTER = re.compile(
    r'(?:' + '|'.join(re.escape(k) for k in BALANCE_KEYWORDS) + r')',
    re.IGNORECASE
)

def filter_balance_messages(df):
    if 'contenu' not in df.columns:
        return df
    mask = df['contenu'].apply(lambda x: bool(BALANCE_RE_FILTER.search(str(x))))
    return df[mask].copy().reset_index(drop=True)

print("Operator detection and balance filter ready.")

Operator detection and balance filter ready.


In [4]:
# ── Amount & balance extraction (from extractor notebook) ──
AMOUNT_RE = re.compile(
    r'(?:montant[^:]*:\s*|(?<!solde\sest\s)de\s+|amount\s+|of\s+)'
    r'(\d[\d\s,\.]*?)\s*(FCFA|XAF)',
    re.IGNORECASE
)
AMOUNT_FALLBACK_RE = re.compile(r'(\d[\d\s]*?)\s*(FCFA|XAF)', re.IGNORECASE)

BALANCE_VAL_RE = re.compile(
    r'(?:'
        r'votre\s+nouveau\s+solde\s+est\s+de\s+|your\s+new\s+balance\s+is\s+|'
        r'nouveau\s+solde\s+est\s+de\s*:?|new\s+balance\s+is\s*:?|'
        r'nouveau\s+solde\s+est\s*:?|new\s+balance\s+is\s*:?|'
        r'nouveau\s+solde\s*:?|new\s+balance\s*:?|'
        r'solde\s*:?|balance\s*:?'
    r')\s*'
    r'(\d[\d\s,\.]*?)\s*(FCFA|XAF)',
    re.IGNORECASE
)

def clean_amount(raw):
    if raw is None:
        return None
    cleaned = re.sub(r'[\s,]', '', str(raw))
    try:
        return float(cleaned)
    except ValueError:
        return None

def extract_amount(text):
    m = AMOUNT_RE.search(text)
    if m:
        return clean_amount(m.group(1)), m.group(2).upper()
    m = AMOUNT_FALLBACK_RE.search(text)
    if m:
        return clean_amount(m.group(1)), m.group(2).upper()
    return None, None

def extract_new_balance(text):
    m = BALANCE_VAL_RE.search(text)
    if m:
        return clean_amount(m.group(1)), m.group(2).upper()
    return None, None

print("Amount & balance extractors ready.")

Amount & balance extractors ready.


In [5]:
# ── Transaction classification rules (from extractor notebook) ──
# Additional rules added per Step 12 (Troubleshooting) for English MoMo patterns
# that were classifying as 'autre': "You have transferred", "You have withdrawn",
# "You have successfully withdrawn", "have via agent...withdrawn",
# "Successful transfer", reversals, vouchers, "Transfert reussi"
TX_RULES = [
    ('retrait', 'OUT', [
        r"retrait\s+d'argent", r'retrait\s+de\s+\d',
        r'vous\s+avez\s+effectue\s+avec\s+succes\s+le\s+retrait',
        r'withdrawal\s+successful', r'cash\s+out',
        r'you\s+have\s+(?:successfully\s+)?withdrawn',
        r'have\s+via\s+agent.*withdrawn',
    ]),
    ('depot', 'IN', [
        r'depot\s+effectue\s+par', r'deposit\s+made\s+by', r'deposit\s+to\s+your',
    ]),
    ('transfert', 'IN', [
        r'(transfert|transfer).*?(frais|fees?)\s*(:|(de)|was|is)?\s*0\s*(xaf|fcfa)',
        r'vous\s+avez\s+re[cç]u\s+\d', r'you\s+have\s+received\s+\d',
        r'has\s+been\s+added\s+to\s+your', r'adjustment\s+has\s+been\s+made',
        r'reversal\s+of\s+\d.*approved',
        r'voucher.*has\s+expired.*new\s+balance',
    ]),
    ('transfert', 'OUT', [
        r'transfert\s+de\s+\d+\s*(fcfa|xaf)', r'transfert\s+de\s+\d+fcfa',
        r'transfer\s+of\s+\d', r'transfert.*effectue.*succes.*\d',
        r'transfert.*vers\s+\d',
        r'you\s+have\s+transferred',
        r'successful\s+transfer',
        r'transfert\s+reussi',
    ]),
    ('paiement', 'OUT', [
        r'paiement\s+de\s+votre\s+facture', r'votre\s+paiement\s+de\s+\d',
        r'your\s+payment\s+of\s+\d', r'paiement.*réussi', r'payment.*successful',
        r'paiement\s+total', r'vous\s+venez\s+d.effectuer\s+un\s+paiement',
        r'voucher.*has\s+been\s+created',
    ]),
    ('rechargement', 'OUT', [
        r'rechargement\s+reussi', r'top.?up\s+successful', r'recharge\s+successful',
    ]),
    ('airtime', 'OUT', [
        r'achete\s+avec\s+succes.*airtime', r'airtime.*transaction',
        r'paiement.*airtime', r'paiement.*de', r'payment.*airtime',
        r'you\s+have\s+received.*airtime\s+from', r'vous\s+avez\s+re[cç]u.*airtime\s+de',
        r'recu.*xaf\s+airtime', r'received.*xaf\s+airtime',
    ]),
    ('transaction', 'OUT', [
        r'une\s+transaction\s+de\s+\d', r'a\s+transaction\s+of\s+\d',
        r'transaction.*effectuee\s+par', r'transaction.*made\s+by',
    ]),
]

TX_RULES_COMPILED = [
    (tx_type, direction, [re.compile(p, re.IGNORECASE) for p in patterns])
    for tx_type, direction, patterns in TX_RULES
]

def classify_transaction(text):
    for tx_type, direction, compiled_patterns in TX_RULES_COMPILED:
        for pattern in compiled_patterns:
            if pattern.search(text):
                return tx_type, direction
    return 'autre', 'unknown'

print("Transaction classifier ready (with extended EN rules).")

Transaction classifier ready (with extended EN rules).


In [6]:
# ── Anonymization engine (from extractor notebook) ──
def short_hash(value, length=4):
    return hashlib.md5(str(value).encode()).hexdigest()[:length].upper()

PHONE_RE = re.compile(r'\b\d{9,12}\b')
NAME_AFTER_PHONE_RE = re.compile(
    r'\b(\d{9,12})\s*[-–:]?\s*([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}'
    r'(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,3})\b'
)
NAME_BEFORE_PHONE_RE = re.compile(
    r'\b([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}'
    r'(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,3})\s*\((\d{9,12})\)'
)
WITHDRAWAL_NAME_RE = re.compile(
    r'\b(?:withdrawn|withdraw|retrait)\b.*?\b(?:chez|at)\s*[:\-–]?\s*'
    r'([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}'
    r'(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){0,4})\b',
    re.IGNORECASE
)

def _phone_variants(phone):
    if not phone:
        return []
    phone = str(phone).strip()
    variants = {phone}
    if re.match(r'^6\d{8}$', phone):
        variants.add('237' + phone)
    if re.match(r'^237\d{9}$', phone):
        variants.add(phone[3:])
    return list(variants)

def anonymize_message(text, user_name=None, user_phone=None, user_id="USER"):
    result = text
    if user_name and user_name.strip():
        result = re.sub(re.escape(user_name.strip()), f'[{user_id}]', result, flags=re.IGNORECASE)

    def replace_withdrawal_name(m):
        name = m.group(1).strip()
        token = short_hash(name)
        return m.group(0).replace(name, f'[CONTACT_{token}]')
    result = WITHDRAWAL_NAME_RE.sub(replace_withdrawal_name, result)

    def replace_name_after(m):
        phone, name = m.group(1), m.group(2).strip()
        return f'{phone} [CONTACT_{short_hash(name)}]'
    result = NAME_AFTER_PHONE_RE.sub(replace_name_after, result)

    def replace_name_before(m):
        name, phone = m.group(1).strip(), m.group(2)
        return f'[CONTACT_{short_hash(name)}] ({phone})'
    result = NAME_BEFORE_PHONE_RE.sub(replace_name_before, result)

    for variant in _phone_variants(user_phone):
        result = result.replace(variant, f'[{user_id}_phone]')

    def replace_phone(m):
        return f'[PHONE_{m.group(0)[-4:]}]'
    result = PHONE_RE.sub(replace_phone, result)

    skip_tokens = {'FCFA', 'XAF', 'SMS', 'ID', 'MTN', 'ORANGE', 'MOBILEMONEY', 'OM', 'MOMO', 'OTP', 'PIN'}
    def replace_caps_name(m):
        name = m.group(0).strip()
        words = name.split()
        if len(words) < 2 or any(w in skip_tokens for w in words):
            return name
        return name
    result = re.sub(
        r'\b([A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}'
        r'(?:\s+[A-ZÀÂÉÈÊËÎÏÔÙÛÜÇ]{2,}){1,3})\b',
        replace_caps_name, result)
    return result

print("Anonymization engine ready.")

Anonymization engine ready.


In [7]:
# ── Core extraction function ──
def extract_transactions_from_df(df, user_id, operator=None):
    """Run the extractor pipeline on a loaded dataframe."""
    if operator is None:
        operator = detect_operator(df)
        if operator == 'Unknown':
            operator = detect_operator_from_content(df)

    df_filtered = filter_balance_messages(df)
    if len(df_filtered) == 0:
        return pd.DataFrame(columns=[
            'UserId', 'Date', 'Heure', 'Operator', 'Transaction_type',
            'Direction', 'Amount', 'Currency', 'New_balance', 'Anonymized_Content'
        ])

    records = []
    for _, row in df_filtered.iterrows():
        text = str(row.get('contenu', ''))
        tx_type, direction = classify_transaction(text)
        amount, currency = extract_amount(text)
        new_balance, bal_currency = extract_new_balance(text)
        if currency is None and bal_currency:
            currency = bal_currency
        anon_content = anonymize_message(text, user_name=None, user_phone=None, user_id=user_id)
        records.append({
            'UserId': user_id, 'Date': row.get('date', ''), 'Heure': row.get('heure', ''),
            'Operator': operator, 'Transaction_type': tx_type, 'Direction': direction,
            'Amount': amount, 'Currency': currency, 'New_balance': new_balance,
            'Anonymized_Content': anon_content,
        })
    return pd.DataFrame(records)

print("Extraction function ready.")

Extraction function ready.


## 2. Step 1 — Data Extraction

Run the extractor on **both** datasets:
- **Original Raw** (51 files in `Data/Original_Raw/`) — raw SMS exports from participants
- **Anonymized Raw** (18 files in `Data/Anonymized_raw/`) — additional anonymized dataset (10 xlsx + 8 csv)

Each file is assigned a unique user ID. Extracted outputs are saved separately.

In [8]:
# ── STEP 1a: Extract from Original Raw ──
all_original = []
file_list = sorted(os.listdir(ORIGINAL_RAW_DIR))
user_counter = 0

for fname in file_list:
    fpath = ORIGINAL_RAW_DIR / fname
    if not (fname.endswith('.xlsx') or fname.endswith('.csv')):
        continue
    user_counter += 1
    user_id = f"orig_user_{user_counter:03d}"
    try:
        df_raw = load_file(fpath)
        df_ext = extract_transactions_from_df(df_raw, user_id)
        if len(df_ext) > 0:
            all_original.append(df_ext)
            print(f"  {fname}: {len(df_raw)} raw -> {len(df_ext)} txns ({user_id})")
        else:
            print(f"  {fname}: 0 balance-related messages ({user_id})")
    except Exception as e:
        print(f"  ERROR {fname}: {str(e)[:80]}")

df_original_extracted = pd.concat(all_original, ignore_index=True) if all_original else pd.DataFrame()
df_original_extracted.to_csv(EXTRACTED_ORIGINAL_DIR / "extracted_original.csv", index=False)
print(f"\nOriginal extraction: {len(df_original_extracted)} transactions from {len(all_original)} files")

  Messages avec MobileMoney 2026-03-21 075833.xlsx: 60 raw -> 15 txns (orig_user_001)


  Messages avec MobileMoney 2026-03-27 201219.xlsx: 56 raw -> 43 txns (orig_user_002)


  Messages with MobileMoney 2026-03-09 073732.xlsx: 257 raw -> 205 txns (orig_user_003)


  Messages with MobileMoney 2026-03-12 045000.xlsx: 873 raw -> 497 txns (orig_user_004)
  Messages with MobileMoney 2026-03-12 132226.xlsx: 76 raw -> 41 txns (orig_user_005)


  Messages with MobileMoney 2026-03-12 200741.xlsx: 121 raw -> 104 txns (orig_user_006)


...

  Messages with MobileMoney 2026-03-28 142533.xlsx: 137 raw -> 73 txns (orig_user_047)
  Messages with MobileMoney 2026-03-28 200505.xlsx: 2 raw -> 2 txns (orig_user_048)


  Messages with MobileMoney 2026-03-28 202952.xlsx: 64 raw -> 59 txns (orig_user_049)


  Messages with MobileMoney 2026-03-29 094220.xlsx: 212 raw -> 118 txns (orig_user_050)


  Messages with OrangeMoney 2026-03-16 150151.xlsx: 113 raw -> 42 txns (orig_user_051)



Original extraction: 19273 transactions from 51 files


In [9]:
# ── STEP 1b: Extract from Anonymized Raw ──
all_anon = []

for fname in sorted(os.listdir(ANONYMIZED_RAW_DIR)):
    fpath = ANONYMIZED_RAW_DIR / fname
    user_id = f"anon_{fname.replace('.xlsx', '').replace('.csv', '')}"

    try:
        if fname.endswith('.xlsx'):
            df_raw = load_file(fpath)
            df_ext = extract_transactions_from_df(df_raw, user_id)
        elif fname.endswith('.csv'):
            df_raw = load_anonymized_csv(fpath)
            operator = detect_operator(df_raw)
            if operator == 'Unknown':
                operator = detect_operator_from_content(df_raw)
            df_filtered = filter_balance_messages(df_raw)
            if len(df_filtered) == 0:
                print(f"  {fname}: {len(df_raw)} raw -> 0 balance msgs ({user_id})")
                continue
            records = []
            for _, row in df_filtered.iterrows():
                text = str(row.get('contenu', ''))
                tx_type, direction = classify_transaction(text)
                amount, currency = extract_amount(text)
                new_balance, bal_currency = extract_new_balance(text)
                if currency is None and bal_currency:
                    currency = bal_currency
                records.append({
                    'UserId': user_id, 'Date': row.get('date', ''), 'Heure': row.get('heure', ''),
                    'Operator': operator, 'Transaction_type': tx_type, 'Direction': direction,
                    'Amount': amount, 'Currency': currency, 'New_balance': new_balance,
                    'Anonymized_Content': text,
                })
            df_ext = pd.DataFrame(records)
        else:
            continue

        if len(df_ext) > 0:
            all_anon.append(df_ext)
            print(f"  {fname}: {len(df_raw)} raw -> {len(df_ext)} txns ({user_id})")
        else:
            print(f"  {fname}: 0 balance msgs ({user_id})")
    except Exception as e:
        print(f"  ERROR {fname}: {str(e)[:80]}")

df_anon_extracted = pd.concat(all_anon, ignore_index=True) if all_anon else pd.DataFrame()
df_anon_extracted.to_csv(EXTRACTED_ANON_DIR / "extracted_anonymized.csv", index=False)
print(f"\nAnonymized extraction: {len(df_anon_extracted)} transactions from {len(all_anon)} files")

  user0001.xlsx: 79 raw -> 54 txns (anon_user0001)


  user0002.xlsx: 69 raw -> 56 txns (anon_user0002)


  user0003.xlsx: 78 raw -> 65 txns (anon_user0003)


  user0004.xlsx: 72 raw -> 54 txns (anon_user0004)


  user0005.xlsx: 67 raw -> 53 txns (anon_user0005)
  user0006.xlsx: 77 raw -> 57 txns (anon_user0006)


  user0007.xlsx: 74 raw -> 52 txns (anon_user0007)
  user0008.xlsx: 71 raw -> 50 txns (anon_user0008)


  user0009.xlsx: 77 raw -> 56 txns (anon_user0009)
  user0010.xlsx: 70 raw -> 50 txns (anon_user0010)
  user011.csv: 17 raw -> 5 txns (anon_user011)
  user012.csv: 108 raw -> 108 txns (anon_user012)


  user013.csv: 220 raw -> 171 txns (anon_user013)
  user014.csv: 9 raw -> 6 txns (anon_user014)
  user015.csv: 16 raw -> 10 txns (anon_user015)
  user016.csv: 203 raw -> 174 txns (anon_user016)


  user017.csv: 27 raw -> 22 txns (anon_user017)
  user018.csv: 279 raw -> 226 txns (anon_user018)



Anonymized extraction: 1269 transactions from 18 files


In [10]:
# ── Post-extraction verification ──
print("=== Extraction Output Verification ===")
print(f"Original:   {df_original_extracted.shape} | Columns: {df_original_extracted.columns.tolist()}")
print(f"Anonymized: {df_anon_extracted.shape}   | Columns: {df_anon_extracted.columns.tolist()}")
print(f"\nColumn match: {set(df_original_extracted.columns) == set(df_anon_extracted.columns)}")
print(f"\nOriginal sample:")
df_original_extracted.head(3)

=== Extraction Output Verification ===
Original:   (19273, 10) | Columns: ['UserId', 'Date', 'Heure', 'Operator', 'Transaction_type', 'Direction', 'Amount', 'Currency', 'New_balance', 'Anonymized_Content']
Anonymized: (1269, 10)   | Columns: ['UserId', 'Date', 'Heure', 'Operator', 'Transaction_type', 'Direction', 'Amount', 'Currency', 'New_balance', 'Anonymized_Content']

Column match: True

Original sample:


,UserId,Date,Heure,Operator,Transaction_type,Direction,Amount,Currency,New_balance,Anonymized_Content
0,orig_user_001,2025-10-07,18:40:27,MobileMoney,transfert,IN,1000.0,XAF,1000.0,Vous avez recu 1000 XAF de [CONTACT_1C67] ([PHONE_0573]) sur votre compte mobile money à 2025-10-07 18:40:26. Messag...
1,orig_user_001,2025-10-08,07:16:40,MobileMoney,paiement,OUT,500.0,XAF,500.0,Votre paiement de 500 XAF a MTNC AIRTIME a ete effectue le 2025-10-08 07:16:25. Votre nouveau solde: 500 XAF. Frais:...
2,orig_user_001,2025-10-08,08:29:15,MobileMoney,transfert,IN,1950.0,XAF,2450.0,Vous avez recu 1950 XAF de MOUSSA ETIENNE SOCIADAM SARL ([PHONE_1001] [CONTACT_8CC4]) sur votre compte Mobile Money ...


## 3. Step 2 — Data Cleaning (Applied Separately)

The **same** cleaning pipeline is applied to both datasets independently:
- Standardize column names
- Convert data types (datetime, numeric)
- Normalize categorical values (lowercase, strip)
- Handle missing values (flagged, not dropped)
- Remove duplicates
- Detect outliers via IQR (flagged, **not** removed)

In [11]:
def clean_dataset(df, label="dataset"):
    """Apply identical cleaning pipeline to a dataset."""
    print(f"--- Cleaning: {label} ({len(df)} rows) ---")
    df = df.copy()
    stats = {'input_rows': len(df)}

    # Standardize column names
    df.columns = [c.strip() for c in df.columns]

    # Convert Date to datetime
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    stats['date_parse_failures'] = int(df['Date'].isna().sum())

    # Convert Heure to string
    df['Heure'] = df['Heure'].astype(str).str.strip()

    # Combine Date + Heure into Datetime
    def combine_dt(row):
        if pd.isna(row['Date']):
            return pd.NaT
        try:
            return pd.to_datetime(f"{row['Date'].strftime('%Y-%m-%d')} {row['Heure']}")
        except Exception:
            return row['Date']
    df['Datetime'] = df.apply(combine_dt, axis=1)

    # Amount to numeric
    df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
    stats['amount_missing'] = int(df['Amount'].isna().sum())

    # New_balance to numeric
    df['New_balance'] = pd.to_numeric(df['New_balance'], errors='coerce')
    stats['balance_missing'] = int(df['New_balance'].isna().sum())

    # Normalize categorical values
    df['Transaction_type'] = df['Transaction_type'].astype(str).str.lower().str.strip()
    df['Direction'] = df['Direction'].astype(str).str.upper().str.strip()
    df['Operator'] = df['Operator'].astype(str).str.strip()
    df['Currency'] = df['Currency'].astype(str).str.upper().str.strip()
    df['Currency'] = df['Currency'].replace({'FCFA': 'XAF', 'NAN': 'XAF', 'NONE': 'XAF'})

    # Missing value summary
    stats['missing_before'] = df.isnull().sum().to_dict()

    # Remove duplicates
    n_before = len(df)
    df = df.drop_duplicates(
        subset=['UserId', 'Date', 'Heure', 'Amount', 'Transaction_type', 'Direction'],
        keep='first'
    )
    stats['duplicates_removed'] = n_before - len(df)

    # Detect outliers using IQR (DO NOT remove)
    if df['Amount'].notna().sum() > 0:
        Q1 = df['Amount'].quantile(0.25)
        Q3 = df['Amount'].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        df['is_outlier'] = (df['Amount'] < lower) | (df['Amount'] > upper)
        stats['outlier_count'] = int(df['is_outlier'].sum())
        stats['iqr_bounds'] = (lower, upper)
    else:
        df['is_outlier'] = False
        stats['outlier_count'] = 0
        stats['iqr_bounds'] = (0, 0)

    # Sort by user and datetime
    df = df.sort_values(['UserId', 'Datetime']).reset_index(drop=True)
    stats['output_rows'] = len(df)
    stats['unique_users'] = df['UserId'].nunique()
    stats['missing_after'] = df.isnull().sum().to_dict()

    print(f"  Date parse failures: {stats['date_parse_failures']}")
    print(f"  Duplicates removed:  {stats['duplicates_removed']}")
    print(f"  Outliers detected:   {stats['outlier_count']} (IQR bounds: {stats['iqr_bounds'][0]:.0f} - {stats['iqr_bounds'][1]:.0f})")
    print(f"  Output: {stats['output_rows']} rows, {stats['unique_users']} users")
    return df, stats

print("Cleaning function defined.")

Cleaning function defined.


In [12]:
# Apply cleaning to BOTH datasets separately
df_orig_clean, orig_stats = clean_dataset(df_original_extracted, "Original")
print()
df_anon_clean, anon_stats = clean_dataset(df_anon_extracted, "Anonymized")

--- Cleaning: Original (19273 rows) ---


  Date parse failures: 0
  Duplicates removed:  12
  Outliers detected:   2789 (IQR bounds: -2500 - 4700)
  Output: 19261 rows, 51 users

--- Cleaning: Anonymized (1269 rows) ---


  Date parse failures: 0
  Duplicates removed:  0
  Outliers detected:   199 (IQR bounds: -88904 - 150840)
  Output: 1269 rows, 18 users


## 4. Step 3 — Add Data Source Labels

Before merging, each dataset receives a `data_source` column to preserve provenance.

In [13]:
df_orig_clean['data_source'] = 'original'
df_anon_clean['data_source'] = 'anonymized'

# Save cleaned pre-merge datasets
df_orig_clean.to_csv(CLEANED_DIR / "cleaned_original.csv", index=False)
df_anon_clean.to_csv(CLEANED_DIR / "cleaned_anonymized.csv", index=False)

print(f"Original:   {len(df_orig_clean)} rows labeled 'original'")
print(f"Anonymized: {len(df_anon_clean)} rows labeled 'anonymized'")
print("Saved to Data/cleaned/")

Original:   19261 rows labeled 'original'
Anonymized: 1269 rows labeled 'anonymized'
Saved to Data/cleaned/


## 5. Step 4 — Dataset Validation

Compare the two cleaned datasets for column consistency, amount distributions, transaction types, and frequency patterns before merging.

In [14]:
print("=== DATASET VALIDATION ===\n")

# Column consistency
orig_cols = set(df_orig_clean.columns)
anon_cols = set(df_anon_clean.columns)
print(f"Columns only in original:   {orig_cols - anon_cols or 'None'}")
print(f"Columns only in anonymized: {anon_cols - orig_cols or 'None'}")

# Amount distributions
print("\n--- Amount Distributions ---")
for name, df in [("Original", df_orig_clean), ("Anonymized", df_anon_clean)]:
    amt = df['Amount'].dropna()
    if len(amt) > 0:
        print(f"  {name}: n={len(amt)}, mean={amt.mean():.0f}, median={amt.median():.0f}, "
              f"std={amt.std():.0f}, min={amt.min():.0f}, max={amt.max():.0f}")

# Transaction counts
print(f"\n--- Transaction Counts ---")
print(f"  Original:   {len(df_orig_clean)} txns, {df_orig_clean['UserId'].nunique()} users")
print(f"  Anonymized: {len(df_anon_clean)} txns, {df_anon_clean['UserId'].nunique()} users")

# Transaction types
print(f"\n--- Transaction Types ---")
orig_types = df_orig_clean['Transaction_type'].value_counts()
anon_types = df_anon_clean['Transaction_type'].value_counts()
all_types = sorted(set(orig_types.index) | set(anon_types.index))
print(f"  {'Type':<20s} | {'Original':>8s} | {'Anonymized':>10s}")
print(f"  {'-'*20}-+-{'-'*8}-+-{'-'*10}")
for t in all_types:
    print(f"  {t:<20s} | {orig_types.get(t, 0):>8d} | {anon_types.get(t, 0):>10d}")

# Direction
print(f"\n--- Direction ---")
for name, df in [("Original", df_orig_clean), ("Anonymized", df_anon_clean)]:
    print(f"  {name}: {dict(df['Direction'].value_counts())}")

# Operator
print(f"\n--- Operator ---")
for name, df in [("Original", df_orig_clean), ("Anonymized", df_anon_clean)]:
    print(f"  {name}: {dict(df['Operator'].value_counts())}")

# Harmonize columns
for col in df_orig_clean.columns:
    if col not in df_anon_clean.columns:
        df_anon_clean[col] = np.nan
for col in df_anon_clean.columns:
    if col not in df_orig_clean.columns:
        df_orig_clean[col] = np.nan

print("\nValidation complete. Datasets are compatible for merging.")

=== DATASET VALIDATION ===

Columns only in original:   None
Columns only in anonymized: None

--- Amount Distributions ---
  Original: n=19261, mean=4431, median=500, std=22201, min=0, max=825575
  Anonymized: n=1269, mean=53345, median=5000, std=93038, min=100, max=348678

--- Transaction Counts ---
  Original:   19261 txns, 51 users
  Anonymized: 1269 txns, 18 users

--- Transaction Types ---
  Type                 | Original | Anonymized
  ---------------------+----------+-----------
  airtime              |     3219 |        283
  depot                |       13 |        195
  paiement             |     2449 |        188
  retrait              |     2244 |        243
  transaction          |     4923 |        181
  transfert            |     6413 |        179

--- Direction ---
  Original: {'OUT': np.int64(15025), 'IN': np.int64(4236)}
  Anonymized: {'OUT': np.int64(895), 'IN': np.int64(374)}

--- Operator ---
  Original: {'MobileMoney': np.int64(19219), 'OrangeMoney': np.int64(42

## 6. Step 5 — Merge Datasets

After validation, concatenate both cleaned datasets. The `data_source` column is preserved to distinguish records.

In [15]:
df_all = pd.concat([df_orig_clean, df_anon_clean], ignore_index=True)

print(f"Merged dataset: {len(df_all)} rows, {df_all['UserId'].nunique()} users")
print(f"Data source distribution: {dict(df_all['data_source'].value_counts())}")
print(f"Columns: {df_all.columns.tolist()}")

# Save cleaned merged
cleaned_path = OUTPUT_DIR / "cleaned_transactions.csv"
df_all.to_csv(cleaned_path, index=False)
print(f"\nSaved: {cleaned_path}")
df_all.head()

Merged dataset: 20530 rows, 69 users
Data source distribution: {'original': np.int64(19261), 'anonymized': np.int64(1269)}
Columns: ['UserId', 'Date', 'Heure', 'Operator', 'Transaction_type', 'Direction', 'Amount', 'Currency', 'New_balance', 'Anonymized_Content', 'Datetime', 'is_outlier', 'data_source']



Saved: C:\Users\ghisl\Documents\DataScience_Final_Project\Data\output\cleaned_transactions.csv


,UserId,Date,Heure,Operator,Transaction_type,Direction,Amount,Currency,New_balance,Anonymized_Content,Datetime,is_outlier,data_source
0,orig_user_001,2025-10-07,18:40:27,MobileMoney,transfert,IN,1000.0,XAF,1000.0,Vous avez recu 1000 XAF de [CONTACT_1C67] ([PHONE_0573]) sur votre compte mobile money à 2025-10-07 18:40:26. Messag...,2025-10-07 18:40:27,False,original
1,orig_user_001,2025-10-08,07:16:40,MobileMoney,paiement,OUT,500.0,XAF,500.0,Votre paiement de 500 XAF a MTNC AIRTIME a ete effectue le 2025-10-08 07:16:25. Votre nouveau solde: 500 XAF. Frais:...,2025-10-08 07:16:40,False,original
2,orig_user_001,2025-10-08,08:29:15,MobileMoney,transfert,IN,1950.0,XAF,2450.0,Vous avez recu 1950 XAF de MOUSSA ETIENNE SOCIADAM SARL ([PHONE_1001] [CONTACT_8CC4]) sur votre compte Mobile Money ...,2025-10-08 08:29:15,False,original
3,orig_user_001,2025-10-08,18:05:40,MobileMoney,paiement,OUT,100.0,XAF,296.0,Votre paiement de 100 XAF a MTNC AIRTIME a ete effectue le 2025-10-08 18:05:25. Votre nouveau solde: 296 XAF. Frais:...,2025-10-08 18:05:40,False,original
4,orig_user_001,2025-10-09,02:19:01,MobileMoney,airtime,OUT,200.0,XAF,96.0,Une transaction de 200 XAF effectuee par MTNC BUNDLES_FORFAITS (MTN_Bundles) sur votre compte d'argent mobile s'es...,2025-10-09 02:19:01,False,original


## 7. Step 6 — Feature Engineering

### Time Features
- year, month, day, weekday, hour, is_weekend

### Behavioral Features (per user, chronological)
- `net_amount`: positive for IN, negative for OUT
- `days_since_last_txn`: gap between consecutive transactions
- `monthly_txn_frequency`: average transactions per month per user
- `send_receive_ratio`: cumulative OUT/IN ratio
- `txn_velocity_7d`: transactions in rolling 7-day window

In [16]:
df_model = df_all.copy()
df_model = df_model.sort_values(['UserId', 'Datetime']).reset_index(drop=True)

# ── Time features ──
df_model['year'] = df_model['Datetime'].dt.year
df_model['month'] = df_model['Datetime'].dt.month
df_model['day'] = df_model['Datetime'].dt.day
df_model['weekday'] = df_model['Datetime'].dt.dayofweek
df_model['hour'] = df_model['Datetime'].dt.hour
df_model['is_weekend'] = df_model['weekday'].isin([5, 6]).astype(int)
print("Time features added: year, month, day, weekday, hour, is_weekend")

Time features added: year, month, day, weekday, hour, is_weekend


In [17]:
# ── net_amount ──
df_model['net_amount'] = df_model.apply(
    lambda r: r['Amount'] if r['Direction'] == 'IN'
              else -r['Amount'] if r['Direction'] == 'OUT'
              else 0, axis=1
)

# ── days_since_last_txn (per user) ──
df_model['days_since_last_txn'] = (
    df_model.groupby('UserId')['Datetime']
    .diff().dt.total_seconds() / 86400
)

# ── monthly_txn_frequency (per user) ──
df_model['year_month'] = df_model['Datetime'].dt.to_period('M')
monthly_counts = df_model.groupby(['UserId', 'year_month']).size().reset_index(name='monthly_txn_count')
user_monthly_avg = monthly_counts.groupby('UserId')['monthly_txn_count'].mean().reset_index(name='monthly_txn_frequency')
df_model = df_model.merge(user_monthly_avg, on='UserId', how='left')

print("Behavioral features added: net_amount, days_since_last_txn, monthly_txn_frequency")

Behavioral features added: net_amount, days_since_last_txn, monthly_txn_frequency


In [18]:
# ── send_receive_ratio (cumulative per user) ──
def compute_send_receive_ratio(group):
    group = group.copy()
    cum_out = (group['Direction'] == 'OUT').cumsum()
    cum_in = (group['Direction'] == 'IN').cumsum()
    group['send_receive_ratio'] = cum_out / cum_in.replace(0, np.nan)
    group['send_receive_ratio'] = group['send_receive_ratio'].fillna(0)
    return group

df_model = df_model.groupby('UserId', group_keys=False).apply(compute_send_receive_ratio)
print("send_receive_ratio added")

send_receive_ratio added


In [19]:
# ── txn_velocity_7d (transactions in past 7 days, per user) ──
def compute_velocity_7d(group):
    group = group.copy().sort_values('Datetime')
    datetimes = group['Datetime'].values
    velocities = []
    for i in range(len(group)):
        current = datetimes[i]
        if pd.isna(current):
            velocities.append(np.nan)
            continue
        window_start = current - pd.Timedelta(days=7)
        count = int(((datetimes[:i+1] >= window_start) & (datetimes[:i+1] <= current)).sum())
        velocities.append(count)
    group['txn_velocity_7d'] = velocities
    return group

df_model = df_model.groupby('UserId', group_keys=False).apply(compute_velocity_7d)

# Drop temp column
df_model = df_model.drop(columns=['year_month'], errors='ignore')

print("txn_velocity_7d added")
print(f"\nFinal feature count: {len(df_model.columns)}")
print(f"Features: {df_model.columns.tolist()}")

txn_velocity_7d added

Final feature count: 24
Features: ['UserId', 'Date', 'Heure', 'Operator', 'Transaction_type', 'Direction', 'Amount', 'Currency', 'New_balance', 'Anonymized_Content', 'Datetime', 'is_outlier', 'data_source', 'year', 'month', 'day', 'weekday', 'hour', 'is_weekend', 'net_amount', 'days_since_last_txn', 'monthly_txn_frequency', 'send_receive_ratio', 'txn_velocity_7d']


## 8. Step 7 — Final Output

Generate two deliverables:
- `cleaned_transactions.csv` — merged, cleaned dataset
- `model_ready_dataset.csv` — with all engineered features

In [20]:
# Save model-ready dataset
model_path = OUTPUT_DIR / "model_ready_dataset.csv"
df_model.to_csv(model_path, index=False)

# Also save a copy in 2_Data_Cleaning for the submission structure
df_all.to_csv(BASE_DIR / "2_Data_Cleaning" / "cleaned_data.csv", index=False)

print(f"Saved: {model_path}")
print(f"Saved: {BASE_DIR / '2_Data_Cleaning' / 'cleaned_data.csv'}")

Saved: C:\Users\ghisl\Documents\DataScience_Final_Project\Data\output\model_ready_dataset.csv
Saved: C:\Users\ghisl\Documents\DataScience_Final_Project\2_Data_Cleaning\cleaned_data.csv


## 9. Pipeline Summary

In [21]:
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"  Total transactions:  {len(df_model)}")
print(f"  Total users:         {df_model['UserId'].nunique()}")
print(f"  Date range:          {df_model['Datetime'].min()} to {df_model['Datetime'].max()}")
print(f"  Features:            {len(df_model.columns)} columns")
print(f"\n  Original stats:")
print(f"    Input rows:        {orig_stats['input_rows']}")
print(f"    After cleaning:    {orig_stats['output_rows']}")
print(f"    Duplicates removed:{orig_stats['duplicates_removed']}")
print(f"    Outliers flagged:  {orig_stats['outlier_count']}")
print(f"    Users:             {orig_stats['unique_users']}")
print(f"\n  Anonymized stats:")
print(f"    Input rows:        {anon_stats['input_rows']}")
print(f"    After cleaning:    {anon_stats['output_rows']}")
print(f"    Duplicates removed:{anon_stats['duplicates_removed']}")
print(f"    Outliers flagged:  {anon_stats['outlier_count']}")
print(f"    Users:             {anon_stats['unique_users']}")
print(f"\n  Transaction types:")
for tx_type, count in df_model['Transaction_type'].value_counts().items():
    print(f"    {tx_type:<20s}: {count}")
print(f"\n  Output files:")
print(f"    - Data/output/cleaned_transactions.csv")
print(f"    - Data/output/model_ready_dataset.csv")
print(f"    - 2_Data_Cleaning/cleaned_data.csv")

PIPELINE SUMMARY
  Total transactions:  20530
  Total users:         69
  Date range:          2023-03-10 21:05:26 to 2026-03-28 23:04:48
  Features:            24 columns

  Original stats:
    Input rows:        19273
    After cleaning:    19261
    Duplicates removed:12
    Outliers flagged:  2789
    Users:             51

  Anonymized stats:
    Input rows:        1269
    After cleaning:    1269
    Duplicates removed:0
    Outliers flagged:  199
    Users:             18

  Transaction types:
    transfert           : 6592
    transaction         : 5104
    airtime             : 3502
    paiement            : 2637
    retrait             : 2487
    depot               : 208

  Output files:
    - Data/output/cleaned_transactions.csv
    - Data/output/model_ready_dataset.csv
    - 2_Data_Cleaning/cleaned_data.csv
